# Loan Default Prediction with Attention-Residual Network

This notebook trains a **tabular Attention-Residual model** for loan default prediction and evaluates it on the same time-based split used by your existing models.

## Goals
- Use origination-time features from `model_data/default_model_data.csv`
- Train an attention + residual neural model for binary default prediction
- Evaluate on validation year (2016) and test year (2017)
- Report the same metrics used in your current pipeline (ROC-AUC, PR-AUC, log loss, Brier score, etc.)

> Tip: If runtime is long on your machine, set `MAX_TRAIN_ROWS` in the config cell to a smaller value for quick experiments.

In [12]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [13]:
# Paths and experiment settings
# Set to a specific Path to force one file; None = pick the first file that exists below.
DATA_PATH_OVERRIDE = None

TARGET = "target_default"

_DATA_CANDIDATES = [
    Path("model_data/default_model_data.csv"),
    Path("model_data/default_model_data_sample.csv"),
    Path("model_data/default_model_data_smoke.csv"),
]


def resolve_data_path() -> Path:
    if DATA_PATH_OVERRIDE is not None:
        p = Path(DATA_PATH_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(f"DATA_PATH_OVERRIDE not found: {p}")
        return p
    for p in _DATA_CANDIDATES:
        if p.exists():
            return p
    raise FileNotFoundError(
        "No dataset found. Expected one of:\n  "
        + "\n  ".join(str(p) for p in _DATA_CANDIDATES)
        + "\nRun prepare_default_model_data.py or add a CSV under model_data/."
    )


DATA_PATH = resolve_data_path()
print(f"Using dataset: {DATA_PATH.resolve()}")

# Use None for full data, or set an integer (e.g., 250_000) for quick iteration.
MAX_TRAIN_ROWS = 250_000
MAX_VALID_ROWS = None
MAX_TEST_ROWS = None

BATCH_SIZE = 1024
EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 4

D_MODEL = 64
NUM_HEADS = 4
DROPOUT = 0.2
FF_MULTIPLIER = 2

Using dataset: /Users/hejiang/Desktop/586-decision-optimization-project/model_data/default_model_data_sample.csv


In [14]:
# Load and split: prefer time-based split (train <=2015, val 2016, test 2017).
# Small/sample files often omit later years; then we fall back to stratified random splits.

df = pd.read_csv(DATA_PATH, low_memory=False)
df = df.dropna(subset=["issue_year"]).copy()
df["issue_year"] = df["issue_year"].astype(int)

train_df = df[df["issue_year"] <= 2015].copy()
valid_df = df[df["issue_year"] == 2016].copy()
test_df = df[df["issue_year"] == 2017].copy()

if train_df.empty or valid_df.empty or test_df.empty:
    print(
        "Time-based split is incomplete for this file (missing years in train/val/test). "
        "issue_year counts:\n",
        df["issue_year"].value_counts().sort_index().to_string(),
        "\n\nUsing stratified 70% / 15% / 15% random split instead "
        "(metrics are not comparable to year-holdout runs until you use full data).\n",
        sep="",
    )
    y_all = df[TARGET].astype(int)
    stratify = y_all if y_all.nunique() > 1 and y_all.value_counts().min() >= 2 else None
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=stratify)
    y_temp = temp_df[TARGET].astype(int)
    stratify_temp = y_temp if y_temp.nunique() > 1 and y_temp.value_counts().min() >= 2 else None
    valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=stratify_temp)

if MAX_TRAIN_ROWS is not None and len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sample(n=MAX_TRAIN_ROWS, random_state=SEED)
if MAX_VALID_ROWS is not None and len(valid_df) > MAX_VALID_ROWS:
    valid_df = valid_df.sample(n=MAX_VALID_ROWS, random_state=SEED)
if MAX_TEST_ROWS is not None and len(test_df) > MAX_TEST_ROWS:
    test_df = test_df.sample(n=MAX_TEST_ROWS, random_state=SEED)

feature_cols = [c for c in train_df.columns if c != TARGET]
categorical_cols = [
    c for c in feature_cols if train_df[c].dtype == "object" or str(train_df[c].dtype).startswith("category")
]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print(f"Train rows: {len(train_df):,}, bad rate: {train_df[TARGET].mean():.4f}")
print(f"Valid rows: {len(valid_df):,}, bad rate: {valid_df[TARGET].mean():.4f}")
print(f"Test rows:  {len(test_df):,}, bad rate: {test_df[TARGET].mean():.4f}")
print(f"Numeric columns: {len(numeric_cols)}, Categorical columns: {len(categorical_cols)}")

Time-based split is incomplete for this file (missing years in train/val/test). issue_year counts:
issue_year
2015    50000

Using stratified 70% / 15% / 15% random split instead (metrics are not comparable to year-holdout runs until you use full data).

Train rows: 35,000, bad rate: 0.2043
Valid rows: 7,500, bad rate: 0.2043
Test rows:  7,500, bad rate: 0.2043
Numeric columns: 14, Categorical columns: 8


In [15]:
# Preprocessing for mixed tabular data


def make_ordinal_encoder() -> OrdinalEncoder:
    """sklearn >= 1.1 supports encoded_missing_value; older versions do not."""
    try:
        return OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1,
        )
    except TypeError:
        return OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)


num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", make_ordinal_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ],
    remainder="drop",
)

x_train_all = preprocessor.fit_transform(train_df[feature_cols])
x_valid_all = preprocessor.transform(valid_df[feature_cols])
x_test_all = preprocessor.transform(test_df[feature_cols])

n_num = len(numeric_cols)
n_cat = len(categorical_cols)

x_train_num = x_train_all[:, :n_num].astype(np.float32)
x_valid_num = x_valid_all[:, :n_num].astype(np.float32)
x_test_num = x_test_all[:, :n_num].astype(np.float32)

# Shift encoded categories by +1 so 0 can represent unknown/missing safely for embeddings.
x_train_cat = x_train_all[:, n_num:].astype(np.int64) + 1
x_valid_cat = x_valid_all[:, n_num:].astype(np.int64) + 1
x_test_cat = x_test_all[:, n_num:].astype(np.int64) + 1

y_train = train_df[TARGET].astype(np.float32).to_numpy()
y_valid = valid_df[TARGET].astype(np.float32).to_numpy()
y_test = test_df[TARGET].astype(np.float32).to_numpy()

# One embedding table per categorical feature.
cat_cardinalities = []
for i in range(n_cat):
    max_id = int(max(x_train_cat[:, i].max(), x_valid_cat[:, i].max(), x_test_cat[:, i].max()))
    cat_cardinalities.append(max_id + 1)

print(f"Numeric matrix shape: {x_train_num.shape}")
print(f"Categorical matrix shape: {x_train_cat.shape}")
print(f"Example cardinalities (first 10): {cat_cardinalities[:10]}")

Numeric matrix shape: (35000, 14)
Categorical matrix shape: (35000, 8)
Example cardinalities (first 10): [3, 12, 4, 4, 3, 13, 50, 3]


In [16]:
class TabularDataset(Dataset):
    def __init__(self, x_num: np.ndarray, x_cat: np.ndarray, y: np.ndarray):
        self.x_num = torch.from_numpy(x_num)
        self.x_cat = torch.from_numpy(x_cat)
        self.y = torch.from_numpy(y).float()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.x_num[idx], self.x_cat[idx], self.y[idx]


class AttentionResidualBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float, ff_multiplier: int = 2):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * ff_multiplier),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_multiplier, d_model),
        )
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(attn_out)

        h = self.ln2(x)
        ff_out = self.ffn(h)
        x = x + self.drop2(ff_out)
        return x


class AttentionResidualNet(nn.Module):
    def __init__(
        self,
        n_num_features: int,
        cat_cardinalities: list[int],
        d_model: int = 64,
        num_heads: int = 4,
        dropout: float = 0.2,
        ff_multiplier: int = 2,
    ):
        super().__init__()
        self.n_cat = len(cat_cardinalities)
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, d_model) for cardinality in cat_cardinalities
        ])

        self.numeric_proj = nn.Sequential(
            nn.Linear(n_num_features, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.block1 = AttentionResidualBlock(d_model, num_heads, dropout, ff_multiplier)
        self.block2 = AttentionResidualBlock(d_model, num_heads, dropout, ff_multiplier)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        num_token = self.numeric_proj(x_num).unsqueeze(1)

        if self.n_cat > 0:
            cat_tokens = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)]
            cat_tokens = torch.stack(cat_tokens, dim=1)
            tokens = torch.cat([num_token, cat_tokens], dim=1)
        else:
            tokens = num_token

        tokens = self.block1(tokens)
        tokens = self.block2(tokens)

        pooled = tokens.mean(dim=1)
        logits = self.head(pooled).squeeze(1)
        return logits

In [17]:
def evaluate_binary(y_true: np.ndarray, proba: np.ndarray, threshold: float = 0.5) -> dict:
    eps = np.finfo(float).eps
    proba = np.clip(proba, eps, 1 - eps)
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

    return {
        "rows": int(len(y_true)),
        "bad_rate": float(np.mean(y_true)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
        "log_loss": float(log_loss(y_true, proba)),
        "brier_score": float(brier_score_loss(y_true, proba)),
        "threshold": threshold,
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


@torch.no_grad()
def predict_proba(model: nn.Module, loader: DataLoader, device: torch.device) -> np.ndarray:
    model.eval()
    probs = []
    for x_num, x_cat, _ in loader:
        x_num = x_num.to(device)
        x_cat = x_cat.to(device)
        logits = model(x_num, x_cat)
        probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)


def run_epoch(model: nn.Module, loader: DataLoader, optimizer, criterion, device: torch.device) -> float:
    model.train()
    running_loss = 0.0
    total = 0

    for x_num, x_cat, y in loader:
        x_num = x_num.to(device)
        x_cat = x_cat.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_num, x_cat)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += float(loss.item()) * batch_size
        total += batch_size

    return running_loss / max(1, total)

In [18]:
train_ds = TabularDataset(x_train_num, x_train_cat, y_train)
valid_ds = TabularDataset(x_valid_num, x_valid_cat, y_valid)
test_ds = TabularDataset(x_test_num, x_test_cat, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = AttentionResidualNet(
    n_num_features=x_train_num.shape[1],
    cat_cardinalities=cat_cardinalities,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
    ff_multiplier=FF_MULTIPLIER,
).to(DEVICE)

n_pos = max(1, int(y_train.sum()))
n_neg = max(1, len(y_train) - n_pos)
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_state = None
best_val_auc = -math.inf
patience_left = PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer, criterion, DEVICE)
    valid_proba = predict_proba(model, valid_loader, DEVICE)
    val_metrics = evaluate_binary(y_valid, valid_proba)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_log_loss": val_metrics["log_loss"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_roc_auc={val_metrics['roc_auc']:.4f} | "
        f"val_pr_auc={val_metrics['pr_auc']:.4f}"
    )

    if val_metrics["roc_auc"] > best_val_auc:
        best_val_auc = val_metrics["roc_auc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_left = PATIENCE
    else:
        patience_left -= 1
        if patience_left == 0:
            print("Early stopping triggered.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df.tail()

Epoch 01 | train_loss=1.0261 | val_roc_auc=0.7129 | val_pr_auc=0.4034
Epoch 02 | train_loss=0.9976 | val_roc_auc=0.7198 | val_pr_auc=0.4071
Epoch 03 | train_loss=0.9899 | val_roc_auc=0.7229 | val_pr_auc=0.4141
Epoch 04 | train_loss=0.9806 | val_roc_auc=0.7255 | val_pr_auc=0.4187
Epoch 05 | train_loss=0.9782 | val_roc_auc=0.7228 | val_pr_auc=0.4150
Epoch 06 | train_loss=0.9759 | val_roc_auc=0.7253 | val_pr_auc=0.4193
Epoch 07 | train_loss=0.9709 | val_roc_auc=0.7287 | val_pr_auc=0.4231
Epoch 08 | train_loss=0.9690 | val_roc_auc=0.7302 | val_pr_auc=0.4258
Epoch 09 | train_loss=0.9714 | val_roc_auc=0.7300 | val_pr_auc=0.4249
Epoch 10 | train_loss=0.9636 | val_roc_auc=0.7291 | val_pr_auc=0.4246
Epoch 11 | train_loss=0.9631 | val_roc_auc=0.7286 | val_pr_auc=0.4246
Epoch 12 | train_loss=0.9616 | val_roc_auc=0.7297 | val_pr_auc=0.4229
Early stopping triggered.


,epoch,train_loss,val_roc_auc,val_pr_auc,val_log_loss
7,8,0.968963,0.730200,0.425786,0.580108
8,9,0.971380,0.730004,0.424910,0.644256
9,10,0.963551,0.729142,0.424646,0.576857
10,11,0.963105,0.728622,0.424574,0.611902
11,12,0.961603,0.729743,0.422893,0.622984


In [19]:
valid_proba = predict_proba(model, valid_loader, DEVICE)
test_proba = predict_proba(model, test_loader, DEVICE)

valid_metrics = evaluate_binary(y_valid, valid_proba)
test_metrics = evaluate_binary(y_test, test_proba)

results_df = pd.DataFrame(
    [
        {"split": "validation_2016", **valid_metrics},
        {"split": "test_2017", **test_metrics},
    ]
)
results_df

,split,rows,bad_rate,roc_auc,pr_auc,log_loss,brier_score,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
0,validation_2016,7500,0.204267,0.730200,0.425786,0.580108,0.198750,0.5,0.690800,0.352897,0.616188,0.448776,4237,1731,588,944
1,test_2017,7500,0.204267,0.713996,0.394560,0.596129,0.205901,0.5,0.673733,0.335963,0.611619,0.433696,4116,1852,595,937


In [20]:
# Optional: save results for reporting/comparison with other models
out_metrics = Path("attention_residual_results.csv")
out_history = Path("attention_residual_training_history.csv")

results_df.to_csv(out_metrics, index=False)
history_df.to_csv(out_history, index=False)

print(f"Saved metrics to: {out_metrics.resolve()}")
print(f"Saved training history to: {out_history.resolve()}")

Saved metrics to: /Users/hejiang/Desktop/586-decision-optimization-project/attention_residual_results.csv
Saved training history to: /Users/hejiang/Desktop/586-decision-optimization-project/attention_residual_training_history.csv
